# Hands-free phone agent on Kaggle (free GPU)

**Before running:**
1. Right panel → **Session options**: Accelerator **GPU T4 x2**, **Internet on** (needs a phone-verified Kaggle account).
2. **Add-ons → Secrets**: add `GITHUB_TOKEN`, a GitHub token that can read the private repo
   (github.com/settings/tokens → fine-grained → only `handsfree-phone-agent`, Contents: read).
3. **Run All**. The phone model and the council's data are copied to `/kaggle/working` at the end,
   so they show up in the **Output** tab.
   To keep results after the session ends, use **Save Version → Save & Run All**.

Rough times on T4: council 1-3 h, training 30-60 min, export + test 15 min. Kaggle sessions stop after 12 h.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
# Work in /kaggle/temp: it is not saved with the notebook, so the token never ends up in Output.
REPO = "/kaggle/temp/handsfree-phone-agent"
URL = f"https://{token}@github.com/Mo3gza21/handsfree-phone-agent.git"
os.makedirs("/kaggle/temp", exist_ok=True)
if not os.path.exists(REPO):
    !git clone -q {URL} {REPO}
    !git -C {REPO} remote set-url origin https://github.com/Mo3gza21/handsfree-phone-agent.git
else:
    !git -C {REPO} pull -q {URL} main
os.chdir(REPO)
!pip install -q -r requirements.txt

## 1. Teachers: your models in Ollama

`MODELS` are pulled from the Ollama library (ollama.com/library). Check the exact Gemma 4 / Qwen 3
tags there. Bigger teachers give better data, and a T4 runs up to ~14B comfortably.
Models that fail to download are skipped.

To use your own GGUF files instead, upload them as a private Kaggle Dataset, attach it with
**Add Input**, and list the paths in `GGUF_FILES`.

In [ ]:
MODELS = ["qwen3:8b", "qwen3:4b", "gemma4:e4b"]   # check tags on ollama.com/library
GGUF_FILES = []   # e.g. ["/kaggle/input/my-models/qwen3-4b-q4_k_m.gguf"]

!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time, pathlib
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
ready = []
for f in GGUF_FILES:
    name = pathlib.Path(f).stem.lower().replace(" ", "-").replace(".", "-")
    pathlib.Path("Modelfile").write_text(f"FROM {f}\n")
    if subprocess.run(["ollama", "create", name, "-f", "Modelfile"]).returncode == 0:
        ready.append(name)
for m in MODELS:
    if subprocess.run(["ollama", "pull", m], stdout=subprocess.DEVNULL).returncode == 0:
        ready.append(m)
    else:
        print("skipped (not found):", m)
MODELS = ready
print("Teachers:", MODELS)
!ollama list

## 2. The council: who teaches best, then rewrite and invent tasks

In [ ]:
models = " ".join(MODELS)
!cd pipeline && python council.py leaderboard --models {models} --n 60 --workers 2

In [ ]:
import json
best = json.load(open("data/leaderboard.json"))["ranking"]
print("Best teacher first:", best)
ordered = " ".join(best)
!cd pipeline && python council.py paraphrase --models {ordered} --n 400 --k 3 --workers 2
!cd pipeline && python council.py invent --models {ordered} --n 300

## 3. Training data

In [ ]:
extra = " ".join(p for p in ["../data/paraphrased.jsonl", "../data/invented.jsonl"] if os.path.exists("data/" + p.split("/")[-1]))
!cd pipeline && python build_dataset.py --sim 5000 --extra {extra}
# Free the GPU for training.
!pkill -f "ollama serve" || true

## 4. Train the phone model

`Qwen/Qwen2.5-1.5B-Instruct` → ~1 GB phone file, Arabic + English. `Qwen/Qwen2.5-0.5B-Instruct` for older phones.

In [ ]:
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
!CUDA_VISIBLE_DEVICES=0 python training/train_lora.py --base {BASE} --batch 8 --grad-accum 2 --epochs 2

## 5. Shrink for the phone

In [ ]:
!python training/export_gguf.py --quant Q4_K_M
!ls -lh outputs/*.gguf

## 6. Test on 150 unseen simulated tasks

If some task types score low, run the last cell and go back to step 4: that's the self-improvement loop.

In [ ]:
import glob
server = next(p for p in glob.glob("outputs/llama.cpp/build/bin/**/llama-server", recursive=True))
srv = subprocess.Popen([server, "-m", "outputs/phone-agent-q4_k_m.gguf", "--port", "8080", "-ngl", "99", "-c", "2048"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(20)
!cd pipeline && python council.py leaderboard --models student@http://localhost:8080/v1 --n 150 --workers 4 --out ../data/student_board.json
srv.terminate()

In [ ]:
# Optional next round, focused on what the student got wrong. Then rerun steps 4-6.
!cd pipeline && python build_dataset.py --sim 5000 --focus ../data/student_board.json --focus-model student --extra {extra}

## 7. Get the phone model

It's in the **Output** tab: `phone-agent-q4_k_m.gguf`. Download it and copy it to the phone.

In [ ]:
!cp outputs/phone-agent-q4_k_m.gguf /kaggle/working/
!cp -r data /kaggle/working/council-data
!ls -lh /kaggle/working/*.gguf